# HW3 · 03 — Fine-tuned multilingual encoder (XLM-RoBERTa-base)
* Trained on the same `train_sub` (40k), evaluated on the same `val` (10k) as the baseline.
* Long texts: training uses **head+tail truncation** (first 256 + last 254 tokens, idea from Sun et al. 2019,
  "How to Fine-Tune BERT for Text Classification?").
* At inference we compare three ways to read long texts on val: head-only, head+tail, and averaging over
  overlapping 512-token chunks.

Kaggle settings: Accelerator **GPU T4 x2**, Internet **On**. Run with *Save Version → Save & Run All*.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding, set_seed)

SEED = 42
MODEL_NAME = "xlm-roberta-base"
MAX_LEN, HEAD = 512, 256           # body = MAX_LEN - 2 special tokens; tail = body - HEAD
EPOCHS, LR, BS = 2, 2e-5, 8        # BS per GPU; with grad accumulation 2 and 2 GPUs -> 32 per step
set_seed(SEED)
print(torch.cuda.device_count(), "GPU(s)", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
TEXT_COL, LABEL_COL = "text", "label"
LANG_COL = next((c for c in ["language", "lang"] if c in train.columns), None)

def make_split(train, label_col="label", lang_col=None, n_val=10_000, n_train=40_000, seed=42):
    strat = train[label_col].astype(str) + ("_" + train[lang_col].astype(str) if lang_col else "")
    rest, val = train_test_split(train, test_size=n_val, stratify=strat, random_state=seed)
    strat_rest = strat.loc[rest.index]
    train_sub, _ = train_test_split(rest, train_size=n_train, stratify=strat_rest, random_state=seed)
    return train_sub.reset_index(drop=True), val.reset_index(drop=True)

train_sub, val = make_split(train, LABEL_COL, LANG_COL)
print(train_sub.shape, val.shape, "first val ids:", val["id"].head(3).tolist())

CLASSES = sorted(train[LABEL_COL].unique())
label2id = {c: i for i, c in enumerate(CLASSES)}
y_tr = train_sub[LABEL_COL].map(label2id).values
y_va = val[LABEL_COL].map(label2id).values

## Tokenisation and the three ways to cut a long text

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
BODY = MAX_LEN - 2

def wrap(x):  # <s> tokens </s>  (CLS/SEP for XLM-R)
    return [tok.cls_token_id] + list(x) + [tok.sep_token_id]

def raw_ids(texts):
    # very long texts: keep first and last 15k characters before tokenising (enough for head and tail)
    texts = [t if len(t) <= 30_000 else t[:15_000] + " " + t[-15_000:] for t in texts]
    return tok(texts, add_special_tokens=False, truncation=False)["input_ids"]

def head_only(ids):
    return [wrap(x[:BODY]) for x in ids]

def head_tail(ids):
    return [wrap(x if len(x) <= BODY else x[:HEAD] + x[-(BODY - HEAD):]) for x in ids]

def chunks(ids, stride=256, max_chunks=8):
    out, owner = [], []
    for i, x in enumerate(ids):
        starts = list(range(0, max(len(x) - BODY, 0) + 1, stride))[:max_chunks]
        if len(x) > BODY and starts[-1] + BODY < len(x) and len(starts) < max_chunks:
            starts.append(len(x) - BODY)  # make sure the end of the text is covered
        for s in starts:
            out.append(wrap(x[s:s + BODY])); owner.append(i)
    return out, np.array(owner)

class DS(torch.utils.data.Dataset):
    def __init__(self, seqs, labels=None):
        self.seqs, self.labels = seqs, labels
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, i):
        d = {"input_ids": self.seqs[i], "attention_mask": [1] * len(self.seqs[i])}
        if self.labels is not None:
            d["labels"] = int(self.labels[i])
        return d

t0 = time.time()
ids_tr = raw_ids(train_sub[TEXT_COL].tolist())
ids_va = raw_ids(val[TEXT_COL].tolist())
ids_te = raw_ids(test[TEXT_COL].tolist())
lens = np.array([len(x) for x in ids_tr])
print(f"tokenised in {time.time() - t0:.0f}s | median tokens {np.median(lens):.0f} | "
      f">510 tokens: {(lens > BODY).mean():.1%} | >2000 tokens: {(lens > 2000).mean():.1%}")

## Fine-tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CLASSES), id2label=dict(enumerate(CLASSES)), label2id=label2id)

def compute_metrics(p):
    return {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}

n_gpu = max(torch.cuda.device_count(), 1)
total_steps = int(np.ceil(len(ids_tr) / (BS * 2 * n_gpu))) * EPOCHS
arg_values = dict(
    output_dir="out", num_train_epochs=EPOCHS, learning_rate=LR, weight_decay=0.01,
    warmup_steps=int(0.06 * total_steps),  # 6% warm-up
    per_device_train_batch_size=BS, gradient_accumulation_steps=2, per_device_eval_batch_size=32,
    fp16=True, eval_strategy="epoch", evaluation_strategy="epoch", save_strategy="no", logging_steps=200,
    group_by_length=True, report_to="none", seed=SEED, dataloader_num_workers=2)
# transformers versions differ in argument names -> keep only the ones this version knows
import inspect
known = set(inspect.signature(TrainingArguments.__init__).parameters)
if "eval_strategy" in known: arg_values.pop("evaluation_strategy")
print("arguments not supported by this transformers version (skipped):", [k for k in arg_values if k not in known])
args = TrainingArguments(**{k: v for k, v in arg_values.items() if k in known})
trainer = Trainer(model=model, args=args, data_collator=DataCollatorWithPadding(tok),
                  train_dataset=DS(head_tail(ids_tr), y_tr), eval_dataset=DS(head_tail(ids_va), y_va),
                  compute_metrics=compute_metrics)
t0 = time.time()
trainer.train()
print(f"training took {(time.time() - t0) / 60:.1f} min")
trainer.save_model("xlmr_model"); tok.save_pretrained("xlmr_model")
pd.DataFrame(trainer.state.log_history).to_csv("train_log.csv", index=False)

## Long-text handling: compare three inference modes on the same val split

In [ ]:
def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def predict(seqs):
    return softmax(trainer.predict(DS(seqs)).predictions)

def predict_mode(ids, mode):
    if mode == "head": return predict(head_only(ids))
    if mode == "headtail": return predict(head_tail(ids))
    seqs, owner = chunks(ids)
    p = predict(seqs)
    return np.stack([p[owner == i].mean(0) for i in range(len(ids))])

val_long = np.array([len(x) > BODY for x in ids_va])
res, val_probs = [], {}
for mode in ["head", "headtail", "chunks"]:
    t0 = time.time()
    p = predict_mode(ids_va, mode); val_probs[mode] = p
    pred = p.argmax(1)
    res.append({"mode": mode, "macro_f1_all": f1_score(y_va, pred, average="macro"),
                "macro_f1_long_only": f1_score(y_va[val_long], pred[val_long], average="macro") if val_long.any() else np.nan,
                "sec": round(time.time() - t0)})
    pd.DataFrame(p, columns=CLASSES).assign(id=val["id"].values).to_csv(f"xlmr_val_proba_{mode}.csv", index=False)
res = pd.DataFrame(res); res.to_csv("long_text_comparison.csv", index=False)
print(f"long texts in val: {val_long.mean():.1%}")
res.round(4)

In [ ]:
best_mode = res.sort_values("macro_f1_all", ascending=False)["mode"].iloc[0]
pred = val_probs[best_mode].argmax(1)
print("best mode:", best_mode)
print(classification_report(y_va, pred, target_names=CLASSES, digits=4))
if LANG_COL:
    for lang in val[LANG_COL].unique():
        m = (val[LANG_COL] == lang).values
        print(f"{lang}: macro-F1 = {f1_score(y_va[m], pred[m], average='macro'):.4f}")

## Test predictions (head+tail and chunks are both saved for the ensemble notebook)

In [ ]:
test_probs = {}
for mode in ["headtail", "chunks"]:
    test_probs[mode] = predict_mode(ids_te, mode)
    pd.DataFrame(test_probs[mode], columns=CLASSES).assign(id=test["id"].values).to_csv(f"xlmr_test_proba_{mode}.csv", index=False)

use = best_mode if best_mode in test_probs else "headtail"
sub = pd.DataFrame({"id": test["id"], "label": np.array(CLASSES)[test_probs[use].argmax(1)]})
sub = sample_sub[["id"]].merge(sub, on="id", how="left")
assert sub["label"].notna().all() and len(sub) == len(sample_sub)
sub.to_csv("submission_xlmr.csv", index=False)
print("submission from mode:", use); print(sub["label"].value_counts(normalize=True).round(3))